In [ ]:
import asyncio
import re
import urllib.parse
from typing import Dict, Any
from playwright.async_api import async_playwright, BrowserContext, Page
from pprint import pprint

async def fetch_pchome(context: BrowserContext, keyword: str) -> Dict[str, Any]:
    result = {"platform": "PChome 24h", "title": "未找到相關商品", "price": 0, "url": "", "status": "無結果"}
    encoded_kw = urllib.parse.quote(keyword)
    api_url = f"https://ecshweb.pchome.com.tw/search/v3.3/all/results?q={encoded_kw}&page=1"
    try:
        response = await context.request.get(api_url, timeout=10000)
        if response.status == 200:
            data = await response.json()
            prods = data.get("prods", [])
            if prods:
                item = prods[0]
                result["title"] = item.get("name", "未知的商品標題")
                result["price"] = int(item.get("price", 0))
                result["url"] = f"https://24h.pchome.com.tw/prod/{item.get('Id', '')}"
                result["status"] = "成功"
    except Exception:
        pass
    return result

async def fetch_momo(context: BrowserContext, keyword: str) -> Dict[str, Any]:
    result = {"platform": "momo購物網", "title": "未找到相關商品", "price": 0, "url": "", "status": "無結果"}
    encoded_kw = urllib.parse.quote(keyword)
    url = f"https://www.momoshop.com.tw/search/searchShop.jsp?keyword={encoded_kw}"
    page: Page = await context.new_page()
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=15000)
        await page.wait_for_timeout(1000)
        cards = page.locator("div.listArea ul li, .prdListArea ul li")
        if await cards.count() > 0:
            card = cards.first
            title_loc = card.locator(".prdName, h3, .goodsName")
            price_loc = card.locator(".price, .money, .prdPrice")
            link_loc = card.locator("a.goods-img-url, a.prdName, a").first
            title = await title_loc.first.inner_text() if await title_loc.count() > 0 else ""
            price_text = await price_loc.first.inner_text() if await price_loc.count() > 0 else ""
            href = await link_loc.get_attribute("href") if await link_loc.count() > 0 else ""
            digits = re.sub(r"[^\d]", "", price_text)
            price = int(digits) if digits else 0
            if href and not href.startswith("http"):
                href = f"https://www.momoshop.com.tw{href}"
            if title:
                result["title"] = title.strip()
                result["price"] = price
                result["url"] = href
                result["status"] = "成功"
    except Exception:
        pass
    finally:
        await page.close()
    return result

async def fetch_yahoo(context: BrowserContext, keyword: str) -> Dict[str, Any]:
    result = {"platform": "Yahoo購物中心", "title": "未找到相關商品", "price": 0, "url": "", "status": "無結果"}
    encoded_kw = urllib.parse.quote(keyword)
    url = f"https://tw.buy.yahoo.com/search/product?p={encoded_kw}"
    page: Page = await context.new_page()
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=15000)
        await page.wait_for_timeout(1200)
        cards = page.locator("a[href*='/gdsale/']")
        if await cards.count() > 0:
            card = cards.first
            href = await card.get_attribute("href")
            txt = await card.inner_text()
            lines = [l.strip() for l in txt.split("\n") if l.strip()]
            title, price = "", 0
            for l in lines:
                if l.startswith("$"):
                    digits = re.sub(r"[^\d]", "", l)
                    if digits and price == 0:
                        price = int(digits)
                elif l not in ["比較", "找相似", "活動", "券", "限時下殺", "折扣"] and not title:
                    title = l
            if title:
                result["title"] = title
                result["price"] = price
                result["url"] = href or ""
                result["status"] = "成功"
    except Exception:
        pass
    finally:
        await page.close()
    return result

# 新增：跨賣場併發查詢函式
async def fetch_item_across_platforms(context: BrowserContext, brand: str, name: str, keyword: str) -> Dict[str, Any]:
    stores_tasks = [
        fetch_pchome(context, keyword),
        fetch_momo(context, keyword),
        fetch_yahoo(context, keyword)
    ]
    store_results = await asyncio.gather(*stores_tasks)
    return {
        "brand": brand,
        "name": name,
        "keyword": keyword,
        "stores": store_results
    }

# 單獨測試跨賣場併發
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context(user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36")
    res = await fetch_item_across_platforms(context, "毛寶", "毛寶 貼身衣物手洗精 1000g", "毛寶 貼身衣物手洗精")
    pprint(res)
    await browser.close()